<a href="https://colab.research.google.com/github/Xinyi-Christie-Dong/DubsTech-Datathon-2026/blob/main/imt574_final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

0. Overview
1. Dataset Description
2. Research Question
3. Why XGBoost? (the model is undecided)
4. Model: XGBoost Classifier
    4.1 Feature Engineering
    4.2 Train / Test Split
    4.3 Train XGBoost
    4.4 Evaluation
    4.5 Feature Importance
5. Hyperparameter Sensitivity Analysis
    5.1 Varying n_estimators
    5.2 Results
6. Model Evaluation
    6.1 Classification Report
    6.2 How Evaluation Changed with Hyperparameter Selection
7. Visualization
8. Challenges
9. How the Model Answers the Research Question
    9.1 Risks and Advantages on Unseen Data
    9.2 Consequences of Being Wrong
    9.3 Societal Impacts
10. Future Work
11. Model Card

# Simplified Catan Simulator & ML Pipeline

## 0. Overview
This project generates training data from a simplified *Settlers of Catan* simulator
to train machine learning models, such as XGBoos, to evaluate and predict game decisions.

The simulator runs 1,000-5,000 randomized single-player games, recording every step as a
`(state, action, reward, next_state, done)` tuple saved to CSV. Each row captures a full
board snapshot before and after an action, including terrain, road placement, vertex
occupancy, player resources, and victory points. This dataset is then used to train a
model to approximate a value function over game decisions.

## Simplifications from Original Catan

To keep the simulator tractable for data generation, several rules from the original game are simplified or removed:

- **Single player**: the game is played by one player only, with no opponents, trading, or competitive blocking
- **Smaller board**: a 7-tile hexagonal grid (radius 1) replaces the standard 19-tile board, reducing the number of vertices and edges significantly
- **Lower victory point threshold**: the game ends at **5 victory points** instead of the standard 10
- **No trading**: there is no port trading, bank trading, or player-to-player trading; all resources come solely from dice rolls
- **No robber / desert tile**: rolling a 7 does not trigger the robber mechanic, and no desert tile is included
- **No development cards**: knight cards, victory point cards, and progress cards are all excluded
- **No largest army / longest road bonuses**: these special victory point awards are not tracked
- **Free starting resources**: after the opening placement, the player is given 2 of each resource to ensure the game can always progress, rather than deriving resources from the second opening settlement as in standard Catan
- **One opening placement**: the player places only one settlement and one road at the start, rather than the standard two placements in opposite directions
- **Random strategy only**: the player always acts randomly (within legal moves), with no human or AI decision-making during data generation
- **Turn limit:** Since the player acts randomly with no strategic direction, some games could theoretically run indefinitely without ever reaching 5 VP. A hard cap of 200 turns is imposed per game to prevent infinite loops during simulation, ensuring the data generation pipeline always terminates in a reasonable time, any game that has not been won by turn 200 is ended and recorded as a loss (`done = False`).

In [ ]:
# import modules
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [ ]:
# load catan_data.csv from google drive
from google.colab import drive
drive.mount('/content/drive')

data = pd.read_csv('/content/drive/MyDrive/Catan-Machine-Learning/catan_data.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data.head()

,game_id,step_id,state_victory_points,state_num_roads,state_num_settlements,state_num_cities,state_res_wood,state_res_brick,state_res_sheep,state_res_wheat,...,next_res_wood,next_res_brick,next_res_sheep,next_res_wheat,next_res_ore,next_tiles_json,next_vertices_json,next_edges_json,reward,done
0,0,0,0,0,0,0,0,0,0,0,...,2,2,2,2,2,"{""(-1, 0, 1)"": {""terrain_type"": ""wheat"", ""dice...","{""[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"": {""has_...","{""[(-1, 0, 1), (0, -1, 1)]"": {""has_road"": fals...",1,False
1,0,1,1,1,1,0,2,2,2,2,...,2,1,2,2,2,"{""(-1, 0, 1)"": {""terrain_type"": ""wheat"", ""dice...","{""[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"": {""has_...","{""[(-1, 0, 1), (0, -1, 1)]"": {""has_road"": fals...",0,False
2,0,2,1,2,1,0,2,1,2,2,...,1,0,2,3,2,"{""(-1, 0, 1)"": {""terrain_type"": ""wheat"", ""dice...","{""[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"": {""has_...","{""[(-1, 0, 1), (0, -1, 1)]"": {""has_road"": fals...",0,False
3,0,3,1,3,1,0,1,0,2,3,...,2,0,2,3,2,"{""(-1, 0, 1)"": {""terrain_type"": ""wheat"", ""dice...","{""[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"": {""has_...","{""[(-1, 0, 1), (0, -1, 1)]"": {""has_road"": fals...",0,False
4,0,4,1,3,1,0,2,0,2,3,...,2,0,2,3,2,"{""(-1, 0, 1)"": {""terrain_type"": ""wheat"", ""dice...","{""[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"": {""has_...","{""[(-1, 0, 1), (0, -1, 1)]"": {""has_road"": fals...",0,False


# 1. Descriptive statistics

## Dataset Schema

Each row in the CSV represents one step (action) taken during a game.

### Identifiers
| Column | Description |
|--------|-------------|
| `game_id` | Unique ID for each simulated game |
| `step_id` | Step number within the game (0 = opening placement) |

### State & Next State
Columns prefixed with `state_` capture the board before the action;
`next_` columns capture the board after. Both share the same fields:

| Column | Description |
|--------|-------------|
| `*_victory_points` | Player's current victory points |
| `*_num_roads` | Number of roads built |
| `*_num_settlements` | Number of settlements built |
| `*_num_cities` | Number of cities built |
| `*_res_wood` | Wood resources held |
| `*_res_brick` | Brick resources held |
| `*_res_sheep` | Sheep resources held |
| `*_res_wheat` | Wheat resources held |
| `*_res_ore` | Ore resources held |
| `*_tiles_json` | JSON map of each tile's terrain type and dice number |
| `*_vertices_json` | JSON map of each vertex's settlement/city status and owner |
| `*_edges_json` | JSON map of each edge's road status and owner |

### Action
| Column | Description |
|--------|-------------|
| `action_dice_roll` | Dice roll value (2–12); `null` on the opening step |
| `action_bought` | What was purchased: `road`, `settlement`, `city`, `settlement+road`, or `null` if nothing was bought |
| `action_location` | Board location (vertex or edge key) where the piece was placed; `null` if nothing was bought |
| `action_resources_gained` | JSON dict of resources collected from the dice roll, e.g. `{"wood": 1}` |

### Outcome
| Column | Description |
|--------|-------------|
| `reward` | Change in victory points this step (0 or +1) |
| `done` | `True` if the player reached 5 VP and won; `False` otherwise |

In [ ]:
# Basic Shape

print(f"Rows: {data.shape[0]:,}  |  Columns: {data.shape[1]}")

Rows: 185,733  |  Columns: 32


In [ ]:
# Data Types & Missing Values

data.info()
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185733 entries, 0 to 185732
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   game_id                  185733 non-null  int64  
 1   step_id                  185733 non-null  int64  
 2   state_victory_points     185733 non-null  int64  
 3   state_num_roads          185733 non-null  int64  
 4   state_num_settlements    185733 non-null  int64  
 5   state_num_cities         185733 non-null  int64  
 6   state_res_wood           185733 non-null  int64  
 7   state_res_brick          185733 non-null  int64  
 8   state_res_sheep          185733 non-null  int64  
 9   state_res_wheat          185733 non-null  int64  
 10  state_res_ore            185733 non-null  int64  
 11  state_tiles_json         185733 non-null  object 
 12  state_vertices_json      185733 non-null  object 
 13  state_edges_json         185733 non-null  object 
 14  acti

,0
game_id,0
step_id,0
state_victory_points,0
state_num_roads,0
state_num_settlements,0
state_num_cities,0
state_res_wood,0
state_res_brick,0
state_res_sheep,0
state_res_wheat,0


In [ ]:
# Game-Level Statistics
game_stats = data.groupby('game_id').agg(
    total_steps   = ('step_id', 'max'),
    won           = ('done', 'any')
).reset_index()

print(f"Total games      : {len(game_stats):,}")
print(f"Games won        : {game_stats['won'].sum():,}")
print(f"Win rate         : {game_stats['won'].mean():.1%}")
print(f"Avg steps / game : {game_stats['total_steps'].mean():.1f}")
print(f"Max steps / game : {game_stats['total_steps'].max()}")

Total games      : 1,000
Games won        : 112
Win rate         : 11.2%
Avg steps / game : 184.7
Max steps / game : 200


In [ ]:
# Action Distribution

data['action_bought'].value_counts(dropna=False)

,count
action_bought,
NaN,173591
road,9668
settlement+road,1000
settlement,846
city,628


In [ ]:
# Reward Distribution

reward_counts = data['reward'].value_counts().sort_index()
reward_pct = (reward_counts / len(data) * 100).round(2)

pd.DataFrame({
    'count':      reward_counts,
    'percentage': reward_pct
})

,count,percentage
reward,,
0,183259,98.67
1,2474,1.33


## 2. Research Question

**What are we trying to predict?**

This is a **supervised learning** problem. We aim to predict the `reward` (change in victory
points) at each game step given the current board state and the action taken. More specifically:

> *Given the player's current resources, board position (roads, settlements, cities), and the
> action taken this turn (dice roll and purchase decision), can we predict whether the player
> will gain a victory point on this step?*

Since `reward` is binary, either 0 (no VP gained) or 1 (VP gained), this is framed as a
**binary classification** task.

**What are we trying to accomplish?**

The long-term goal is to train a model that can evaluate the quality of a decision at any
point in the game. By learning which combinations of state and action are associated with
earning a victory point, the model approximates a value function over game decisions. This
lays the groundwork for identifying stronger strategies, for example, understanding whether
building a road, settlement, or city at a given state is more likely to advance the player
toward winning.

## 3. Why XGBoost?

We use **XGBoost (Extreme Gradient Boosting)** to answer this research question for several reasons:

- **Handles class imbalance well**: The reward signal is highly sparse, only 1.33% of steps
  earn a VP. XGBoost supports `scale_pos_weight` to adjust for this imbalance without
  requiring resampling.

- **Works well with tabular data**: Our features are structured numeric columns (resources,
  road counts, VP, dice rolls). XGBoost is consistently one of the strongest performers on
  tabular data.

- **Captures non-linear interactions**: The relationship between game state and a VP reward
  is non-linear, for example, having wheat and ore only leads to a VP if a city can be
  built. Tree-based models like XGBoost naturally capture these conditional interactions
  without manual feature engineering.

- **Feature importance**: XGBoost provides built-in feature importance scores, letting us
  interpret which resources or board features most influence whether a VP is earned, useful
  both for model understanding and for the game strategy analysis.

- **Efficient on large datasets**: With ~185,000 rows, training speed matters. XGBoost is
  highly optimized and handles this scale efficiently.

## 4. Model: XGBoost Classifier

### 4.1 Feature Engineering

We use three groups of features:

- **Numeric state features**: the player's current resources (wood, brick, sheep, wheat, ore),
  road/settlement/city counts, victory points, and the dice roll this turn
- **Tile features**: for each terrain type, the number of tiles on the board (`tile_count_*`)
  and the total pip sum (`tile_pipsum_*`) ,capturing how frequently each resource is produced
  per dice roll
- **Vertex features**: for each player-owned settlement or city, we compute an income score
  (sum of pip counts of adjacent tiles) and terrain diversity (number of distinct terrain types
  touching that vertex), then aggregate across all owned vertices, capturing how well-positioned
  the player is to collect resources

The target variable is `reward` (0 = no VP gained, 1 = VP gained this step).

In [ ]:
# Constants

# Pip count: probability weight for each dice number (out of 36 rolls)
# Reflects the bell-curve distribution of rolling two six-sided dice
PIPS = {2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 5, 9: 4, 10: 3, 11: 2, 12: 1}

# Resource types tracked throughout
RESOURCES = ["wheat", "sheep", "forest", "ore", "brick"]

# Map terrain name to resource name (forest produces wood)
TERRAIN_TO_RESOURCE = {
    "forest": "wood",
    "wheat":  "wheat",
    "sheep":  "sheep",
    "ore":    "ore",
    "brick":  "brick",
}

PLAYER_ID = 1

In [ ]:
# 1. Tiles features

def parse_tiles(tiles_json_str):
    """
    Parse the tiles JSON and compute per-resource pip sums and tile counts.

    For each terrain type, we compute:
      - tile_count_<terrain>  : number of tiles of that type on the board
      - tile_pipsum_<terrain> : sum of pip counts of all tiles of that type
                                = expected activations per 36 rolls

    Example (from the board above):
      wheat tile with dice_number=9  -> pips=4
      brick tile with dice_number=11 -> pips=2
      brick tile with dice_number=9  -> pips=4
      -> tile_pipsum_brick = 2 + 4 = 6  (brick activates 6 out of every 36 rolls)

    Args:
        tiles_json_str : JSON string from the state_tiles_json column

    Returns:
        dict of {feature_name: value}
    """
    tiles = json.loads(tiles_json_str)

    counts  = {t: 0 for t in RESOURCES}
    pipsums = {t: 0 for t in RESOURCES}

    for coord_str, info in tiles.items():
        terrain     = info["terrain_type"]
        dice_number = info.get("dice_number") or 0

        if terrain in counts:
            counts[terrain]  += 1
            pipsums[terrain] += PIPS.get(dice_number, 0)

    features = {}
    for terrain in RESOURCES:
        features[f"tile_count_{terrain}"]  = counts[terrain]
        features[f"tile_pipsum_{terrain}"] = pipsums[terrain]

    return features

In [ ]:
# 2. Vertices features

def _parse_vertex_key(vertex_key_str):
    """
    Convert the string vertex key back into a list of tile coordinate tuples.

    The key is stored as e.g. "[(-1, 0, 1), (0, -1, 1), (0, 0, 0)]"
    which represents the 3 tile coordinates that meet at this vertex corner.

    Returns:
        list of (x, y, z) tuples
    """
    return ast.literal_eval(vertex_key_str)


def parse_vertices(vertices_json_str, tiles_json_str, player_id=PLAYER_ID):
    """
    Parse the vertices JSON and compute placement quality features for
    the player's owned settlements and cities.

    For each player-owned vertex we compute:
      - terrain diversity : number of distinct terrain types on adjacent tiles
                            higher = more varied resources per roll
      - income score      : sum of pip counts of adjacent tiles
                            higher = more frequent resource collection

    Then we aggregate across all owned vertices:
      - vertex_owned_count   : total number of owned vertices (settlements + cities)
      - vertex_total_income  : sum of income scores across all owned vertices
                               the main proxy for how often the player gets resources
      - vertex_avg_income    : average income score per owned vertex
                               isolates placement quality from quantity
      - vertex_max_diversity : highest terrain diversity across all owned vertices
                               reflects how varied the player's resource access is
      - vertex_total_diversity : sum of diversity scores across all owned vertices

    Args:
        vertices_json_str : JSON string from the state_vertices_json column
        tiles_json_str    : JSON string from the state_tiles_json column
                            (needed to look up terrain type and dice number per tile)
        player_id         : the player whose vertices to analyse (default 1)

    Returns:
        dict of {feature_name: value}
    """
    vertices  = json.loads(vertices_json_str)
    tiles     = json.loads(tiles_json_str)

    # Build a lookup: "(x, y, z)" string -> {terrain_type, dice_number}
    tile_lookup = {k: v for k, v in tiles.items()}

    income_scores    = []
    diversity_scores = []

    for vertex_key_str, vdata in vertices.items():
        if vdata.get("owner") != player_id:
            continue

        # Parse the vertex key to get the 3 adjacent tile coordinates
        adj_tiles = _parse_vertex_key(vertex_key_str)

        terrain_types = set()
        income        = 0

        for tile_coord in adj_tiles:
            # Convert tuple to the same string format used in tiles_json
            tile_key = str(tile_coord)
            if tile_key not in tile_lookup:
                continue   # tile is outside the board (border vertex)

            info        = tile_lookup[tile_key]
            terrain     = info["terrain_type"]
            dice_number = info.get("dice_number") or 0

            terrain_types.add(terrain)
            income += PIPS.get(dice_number, 0)

        income_scores.append(income)
        diversity_scores.append(len(terrain_types))

    # Aggregate — return zeros if player owns no vertices yet
    n = len(income_scores)
    return {
        "vertex_owned_count":     n,
        "vertex_total_income":    sum(income_scores)    if n else 0,
        "vertex_avg_income":      sum(income_scores) / n if n else 0,
        "vertex_max_diversity":   max(diversity_scores) if n else 0,
        "vertex_total_diversity": sum(diversity_scores) if n else 0,
    }

In [ ]:
# 3. Edges features
'''
def parse_edges(edges_json_str, player_id=PLAYER_ID):
    """
    Parse the edges JSON and compute road network features.

    Features:
      - edge_roads_built   : number of roads the player has built
                             (confirms the numeric state_ column)
      - edge_frontier_count: number of empty edges adjacent to the player's
                             road network — measures expansion potential.
                             An edge is a "frontier" if:
                               (a) it has no road yet, AND
                               (b) it shares at least one tile with an edge
                                   that the player already owns

    Args:
        edges_json_str : JSON string from the state_edges_json column
        player_id      : the player whose roads to analyse (default 1)

    Returns:
        dict of {feature_name: value}
    """
    edges = json.loads(edges_json_str)

    # Parse each edge key into a frozenset of tile coords for adjacency checks
    parsed = {}
    for edge_key_str, edata in edges.items():
        tile_coords = frozenset(
            tuple(c) for c in ast.literal_eval(edge_key_str)
        )
        parsed[edge_key_str] = {
            "tiles":     tile_coords,
            "has_road":  edata["has_road"],
            "owner":     edata["owner"],
        }

    # Collect tile sets of player-owned edges
    player_edge_tiles = set()
    roads_built       = 0
    for edata in parsed.values():
        if edata["owner"] == player_id:
            roads_built += 1
            player_edge_tiles.update(edata["tiles"])

    # Count frontier edges: empty edges that share a tile with the player's network
    frontier_count = 0
    for edata in parsed.values():
        if edata["has_road"]:
            continue
        # An edge is adjacent to the player's network if they share any tile coord
        if edata["tiles"] & player_edge_tiles:
            frontier_count += 1

    return {
        "edge_roads_built":    roads_built,
        "edge_frontier_count": frontier_count,
    }
'''

'\ndef parse_edges(edges_json_str, player_id=PLAYER_ID):\n    """\n    Parse the edges JSON and compute road network features.\n \n    Features:\n      - edge_roads_built   : number of roads the player has built\n                             (confirms the numeric state_ column)\n      - edge_frontier_count: number of empty edges adjacent to the player\'s\n                             road network — measures expansion potential.\n                             An edge is a "frontier" if:\n                               (a) it has no road yet, AND\n                               (b) it shares at least one tile with an edge\n                                   that the player already owns\n \n    Args:\n        edges_json_str : JSON string from the state_edges_json column\n        player_id      : the player whose roads to analyse (default 1)\n \n    Returns:\n        dict of {feature_name: value}\n    """\n    edges = json.loads(edges_json_str)\n \n    # Parse each edge key into a frozenset

In [ ]:
# 4. Apply to a full DataFrame

def extract_json_features(df, prefix="state_"):
    """
    Apply all three parsers to a DataFrame and return a new DataFrame
    of engineered features.

    Args:
        df     : the full catan_data DataFrame
        prefix : column prefix to read from, either "state_" or "next_"

    Returns:
        DataFrame with all engineered features aligned to df's index
    """
    tiles_col    = f"{prefix}tiles_json"
    vertices_col = f"{prefix}vertices_json"
    # edges_col    = f"{prefix}edges_json"

    print(f"Parsing {tiles_col} ...")
    tile_feats = df[tiles_col].apply(parse_tiles).apply(pd.Series)

    print(f"Parsing {vertices_col} ...")
    vertex_feats = df.apply(
        lambda row: parse_vertices(row[vertices_col], row[tiles_col]),
        axis=1
    ).apply(pd.Series)

    #print(f"Parsing {edges_col} ...")
    #edge_feats = df[edges_col].apply(parse_edges).apply(pd.Series)

    # Prefix all new columns so state_ and next_ features stay distinct
    tile_feats.columns   = [f"{prefix}{c}" for c in tile_feats.columns]
    vertex_feats.columns = [f"{prefix}{c}" for c in vertex_feats.columns]
    #edge_feats.columns   = [f"{prefix}{c}" for c in edge_feats.columns]

    return pd.concat([tile_feats, vertex_feats], axis=1)

In [ ]:
data_sample = data.head(10).reset_index(drop=True)

In [ ]:
all_json_features = []

for prefix in ["state_", "next_"]:
    print(f"\nExtracting features for prefix: {prefix}")
    json_feats = extract_json_features(data, prefix=prefix)
    all_json_features.append(json_feats)

json_features_df = pd.concat(all_json_features, axis=1)
data_enriched = pd.concat([data, json_features_df], axis=1)

print(f"\nOriginal columns : {data.shape[1]}")
print(f"New JSON features: {json_features_df.shape[1]}")
print(f"Enriched columns : {data_enriched.shape[1]}")
data_enriched.head()


Extracting features for prefix: state_
Parsing state_tiles_json ...
Parsing state_vertices_json ...

Extracting features for prefix: next_
Parsing next_tiles_json ...
Parsing next_vertices_json ...

Original columns : 32
New JSON features: 30
Enriched columns : 62


,game_id,step_id,state_victory_points,state_num_roads,state_num_settlements,state_num_cities,state_res_wood,state_res_brick,state_res_sheep,state_res_wheat,...,next_tile_pipsum_forest,next_tile_count_ore,next_tile_pipsum_ore,next_tile_count_brick,next_tile_pipsum_brick,next_vertex_owned_count,next_vertex_total_income,next_vertex_avg_income,next_vertex_max_diversity,next_vertex_total_diversity
0,0,0,0,0,0,0,0,0,0,0,...,12,1,4,2,8,1.0,15.0,15.0,3.0,3.0
1,0,1,1,1,1,0,2,2,2,2,...,12,1,4,2,8,1.0,15.0,15.0,3.0,3.0
2,0,2,1,2,1,0,2,1,2,2,...,12,1,4,2,8,1.0,15.0,15.0,3.0,3.0
3,0,3,1,3,1,0,1,0,2,3,...,12,1,4,2,8,1.0,15.0,15.0,3.0,3.0
4,0,4,1,3,1,0,2,0,2,3,...,12,1,4,2,8,1.0,15.0,15.0,3.0,3.0


In [ ]:
data_enriched.head(1).T

,0
game_id,0
step_id,0
state_victory_points,0
state_num_roads,0
state_num_settlements,0
...,...
next_vertex_owned_count,1.0
next_vertex_total_income,15.0
next_vertex_avg_income,15.0
next_vertex_max_diversity,3.0


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

# --- Feature selection ---
NUMERIC_COLS = [
    'state_victory_points',
    'state_num_roads',
    'state_num_settlements',
    'state_num_cities',
    'state_res_wood',
    'state_res_brick',
    'state_res_sheep',
    'state_res_wheat',
    'state_res_ore',
    'action_dice_roll',
]

TILE_COLS = [f"state_tile_count_{t}" for t in ["wheat", "sheep", "forest", "ore", "brick"]] + \
            [f"state_tile_pipsum_{t}" for t in ["wheat", "sheep", "forest", "ore", "brick"]]

VERTEX_COLS = [
    'state_vertex_owned_count',
    'state_vertex_total_income',
    'state_vertex_avg_income',
    'state_vertex_max_diversity',
    'state_vertex_total_diversity',
]

FEATURE_COLS = NUMERIC_COLS + TILE_COLS + VERTEX_COLS

print(f"Total features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

TARGET = 'reward'

df_model = data_enriched[FEATURE_COLS + [TARGET]].copy()

# Fill nulls in dice roll (step 0 has no dice roll)
df_model['action_dice_roll'] = df_model['action_dice_roll'].fillna(0)

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")
print(f"Class balance:\n{y.value_counts()}")

Total features: 25
['state_victory_points', 'state_num_roads', 'state_num_settlements', 'state_num_cities', 'state_res_wood', 'state_res_brick', 'state_res_sheep', 'state_res_wheat', 'state_res_ore', 'action_dice_roll', 'state_tile_count_wheat', 'state_tile_count_sheep', 'state_tile_count_forest', 'state_tile_count_ore', 'state_tile_count_brick', 'state_tile_pipsum_wheat', 'state_tile_pipsum_sheep', 'state_tile_pipsum_forest', 'state_tile_pipsum_ore', 'state_tile_pipsum_brick', 'state_vertex_owned_count', 'state_vertex_total_income', 'state_vertex_avg_income', 'state_vertex_max_diversity', 'state_vertex_total_diversity']
Features : 25
Samples  : 185,733
Class balance:
reward
0    183259
1      2474
Name: count, dtype: int64
